In [1]:
#|default_exp llm

In [2]:
#|export
from traitlets import HasTraits, Unicode, List
# from langchain_openai import ChatOpenAI
import os
# from langchain.docstore.document import Document
# from langchain_community.document_loaders import UnstructuredMarkdownLoader
# from langchain_openai import OpenAIEmbeddings
# from langchain_community.document_loaders import PyPDFLoader
# from langchain_community.vectorstores import FAISS
# from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.docstore.document import Document
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [3]:
#|export
class LLM(HasTraits):

    def __init__(self, filepath='GEMINI_API_KEY'):
        super().__init__()

        with open(filepath, 'r') as file:
            gemini_api_key = file.read().strip()
        os.environ['GEMINI_API_KEY'] = gemini_api_key
        self.llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

In [4]:
llm_model = LLM('GEMINI_API_KEY')
llm_model

In [5]:
#|export
class FileModel(LLM):
    # Define a Unicode string trait
    select = Unicode()
    files = List()

    def __init__(self, course_file_dir = 'course_files/'):
        super().__init__()
        self.course_file_dir = course_file_dir
        self.embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
        self.db = None

    def save_content_from_upload(self, values):
        for value in values:
            with open(self.course_file_dir + value['name'], "wb") as fp:
                fp.write(value['content'])

    def load_text_to_db(self, text):
        doc = Document(page_content=text)
        db = FAISS.from_documents(doc, self.embeddings)
        if self.db:
            self.db.merge_from(db)
        else: 
            self.db = db

    def load_pdf_to_db(self, filepath):
        loader = PyPDFLoader(filepath) 
        pages = loader.load_and_split()
        db = FAISS.from_documents(pages, self.embeddings)
        if self.db:
            self.db.merge_from(db)
        else: 
            self.db = db
        self.files.append(filepath)

    def load_markdown_to_db(self, filepath):
        loader = UnstructuredMarkdownLoader(filepath, mode="elements") #mode=elements breaks up the text into chunks
        doc = loader.load()
        db = FAISS.from_documents(doc, self.embeddings)
        if self.db:
            self.db.merge_from(db)
        else: 
            self.db = db
        self.files.append(filepath)

    def save_content_from_upload(values):
        for value in values:
            with open(value['name'], "wb") as fp:
                fp.write(value['content'])

In [6]:
file_model = FileModel()
file_model.load_pdf_to_db("course_files/STP 420 spring 2024 course syllabus.pdf") #file as input

In [7]:
#|hide
import nbdev; nbdev.nbdev_export()

Exception: `nbdev_export` must be called from a directory within a nbdev project.